# Data Exploration: SoundCam RIRs and MedleyDB Input Signals

This notebook visualises the datasets used in the DDSP-ARE experiments:

- **SoundCam** — room impulse responses (RIRs) for two time-varying acoustic scenarios:
  - *Moving listener*: microphone sweeps through 10 positions, speaker is fixed.
  - *Moving person*: a person moves through 100 positions, microphone and speaker are fixed.
- **MedleyDB** — full-track music mixes used as excitation signals.

The notebook produces:
1. Static magnitude-response and IR waveform plots for both scenarios.
2. A schematic room layout showing position indices.
3. Animations of the magnitude response evolving as the position changes.
4. A comparison of the raw vs. interpolated response transition (before/after interpolation).

In [1]:
import sys
from pathlib import Path

root = Path().resolve().parents[1]  # src/scripts -> src -> repo root
sys.path.insert(0, str(root / 'src'))
sys.path.insert(0, str(root / 'src' / 'external'))

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import torch
import torchaudio

from utils.common import load_rirs, get_delay_from_ir, interpolate_IRs
from utils.plotting import configure_text_rendering, log_smooth_curve

configure_text_rendering()
%matplotlib inline
print(f'Repository root: {root}')

Repository root: C:\Users\ferma\PhD\DAFx_26\DDSP-adaptive-EQ-26


---
## 1. Load SoundCam RIRs

In [ ]:
rir_dir_ml = root / 'data' / 'SoundCam' / 'moving_listener'
rir_dir_mp = root / 'data' / 'SoundCam' / 'moving_person'

rirs_ml, srs_ml = load_rirs(rir_dir_ml, max_n=100, normalize=False)
rirs_mp, srs_mp = load_rirs(rir_dir_mp, max_n=100, normalize=False)

sr = srs_ml[0]  # all RIRs share the same sample rate

print(f'Moving listener:  {len(rirs_ml):3d} RIRs  |  sr = {sr} Hz  |  length = {len(rirs_ml[0])} samples ({len(rirs_ml[0])/sr:.2f} s)')
print(f'Moving person:    {len(rirs_mp):3d} RIRs  |  sr = {srs_mp[0]} Hz  |  length = {len(rirs_mp[0])} samples ({len(rirs_mp[0])/sr:.2f} s)')

print('\nDirect-sound delays (moving_listener):')
for i, rir in enumerate(rirs_ml):
    d = get_delay_from_ir(rir, sr)
    print(f'  RIR {i:2d}: delay = {d:5d} samples  ({d/sr*1000:.1f} ms)')

---
## 2. Room Layout — Schematic Position Map

The SoundCam dataset was recorded in a rectangular room (~6 m × 6 m). 
The microphone positions in the *moving listener* scenario are arranged along a horizontal arc, and the person positions in the *moving person* scenario follow a grid sweep. 
Since exact position coordinates are not embedded in the WAV filenames, the cell below draws a schematic using the position **index** as a proxy — index 0 is the first recorded position and index N−1 is the last.

In [ ]:
n_ml = len(rirs_ml)
n_mp = len(rirs_mp)

# Schematic: place positions on a grid / arc for illustration.
# Moving listener — positions along a horizontal row.
ml_x = np.linspace(1.0, 5.0, n_ml)
ml_y = np.full(n_ml, 3.0)

# Moving person — positions on a regular grid.
n_cols_mp = int(np.ceil(np.sqrt(n_mp)))
n_rows_mp = int(np.ceil(n_mp / n_cols_mp))
mp_idx = np.arange(n_mp)
mp_x = 0.5 + (mp_idx % n_cols_mp) * (5.0 / n_cols_mp)
mp_y = 0.5 + (mp_idx // n_cols_mp) * (5.0 / n_rows_mp)

# Fixed positions (shared between scenarios)
speaker_pos = (3.0, 0.5)
mic_fixed   = (3.0, 4.5)  # fixed microphone in moving-person scenario

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

for ax, title, xs, ys, label in [
    (axes[0], 'Moving Listener Scenario', ml_x, ml_y, 'Mic position'),
    (axes[1], 'Moving Person Scenario',   mp_x, mp_y, 'Person position'),
]:
    ax.set_xlim(0, 6)
    ax.set_ylim(0, 6)
    ax.set_aspect('equal')
    ax.add_patch(plt.Rectangle((0, 0), 6, 6, fill=False, edgecolor='black', linewidth=2))

    # Colour points by index (blue → red)
    n = len(xs)
    colours = plt.get_cmap('coolwarm')(np.linspace(0, 1, n))
    ax.scatter(xs, ys, c=colours, s=60, zorder=3, label=label)
    for i in range(n):
        ax.annotate(str(i), (xs[i], ys[i]), fontsize=6, ha='center', va='bottom', xytext=(0, 4), textcoords='offset points')

    ax.plot(*speaker_pos, 'k^', markersize=10, label='Speaker')
    if ax is axes[1]:
        ax.plot(*mic_fixed, 'bs', markersize=9, label='Microphone (fixed)')

    ax.set_xlabel('x [m] (schematic)')
    ax.set_ylabel('y [m] (schematic)')
    ax.set_title(title)
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(True, linestyle=':', alpha=0.5)

fig.suptitle('Schematic Room Layout (position indices, not exact coordinates)', fontsize=11)
fig.tight_layout()
plt.show()

---
## 3. Static Magnitude Response Comparison

Overlay all RIR magnitude spectra in each scenario to visualise the spread of responses across positions.

In [ ]:
NFFT = 2 ** 15
freqs = np.fft.rfftfreq(NFFT, d=1.0 / sr)

def compute_mag_db(rir, nfft=NFFT):
    H = np.fft.rfft(rir, n=nfft)
    return 20.0 * np.log10(np.abs(H) + 1e-8)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, rirs, title in [
    (axes[0], rirs_ml, f'Moving Listener ({len(rirs_ml)} positions)'),
    (axes[1], rirs_mp, f'Moving Person ({len(rirs_mp)} positions)'),
]:
    n = len(rirs)
    colours = plt.get_cmap('coolwarm')(np.linspace(0, 1, n))
    for i, rir in enumerate(rirs):
        mag = compute_mag_db(rir)
        m = freqs > 0
        f_s, mag_s = log_smooth_curve(freqs[m], mag[m], window_pts=201)
        ax.semilogx(f_s, mag_s, color=colours[i], linewidth=0.8, alpha=0.75)

    sm = plt.cm.ScalarMappable(cmap='coolwarm', norm=plt.Normalize(0, n - 1))
    sm.set_array([])
    fig.colorbar(sm, ax=ax, label='Position index')

    ax.set_xlim(50, 20000)
    ax.set_ylim(-60, 10)
    ax.set_xlabel('Frequency [Hz]')
    ax.set_ylabel('Magnitude [dB]')
    ax.set_title(title)
    ax.grid(True, linestyle=':', alpha=0.6)

fig.suptitle('RIR Magnitude Spectra — all positions')
fig.tight_layout()
plt.show()

---
## 4. Animation — Moving Listener Scenario

Animate the magnitude spectrum as the microphone sweeps through all 10 positions.

In [ ]:
def make_mag_animation(rirs, sr, title, nfft=NFFT, fps=4):
    """Return an HTML animation of the magnitude spectrum evolving over position index."""
    freqs_local = np.fft.rfftfreq(nfft, d=1.0 / sr)
    m = freqs_local > 50

    # Pre-compute smoothed spectra for all positions.
    all_f, all_mag = [], []
    for rir in rirs:
        mag = compute_mag_db(rir, nfft=nfft)
        f_s, mag_s = log_smooth_curve(freqs_local[m], mag[m], window_pts=201)
        all_f.append(f_s)
        all_mag.append(mag_s)

    y_min = min(np.min(m_) for m_ in all_mag) - 3
    y_max = max(np.max(m_) for m_ in all_mag) + 3

    fig_a, ax_a = plt.subplots(figsize=(8, 4))
    line, = ax_a.semilogx([], [], linewidth=1.5, color='steelblue')
    ax_a.set_xlim(50, 20000)
    ax_a.set_ylim(y_min, y_max)
    ax_a.set_xlabel('Frequency [Hz]')
    ax_a.set_ylabel('Magnitude [dB]')
    ax_a.grid(True, linestyle=':', alpha=0.6)
    title_text = ax_a.set_title('')

    def init():
        line.set_data([], [])
        return (line,)

    def update(frame):
        line.set_data(all_f[frame], all_mag[frame])
        title_text.set_text(f'{title}  —  Position {frame}')
        return (line, title_text)

    ani = animation.FuncAnimation(
        fig_a, update, frames=len(rirs), init_func=init,
        interval=int(1000 / fps), blit=True,
    )
    plt.close(fig_a)
    return HTML(ani.to_jshtml())

make_mag_animation(rirs_ml, sr, 'Moving Listener')

---
## 5. Animation — Moving Person Scenario

Animate the magnitude spectrum across all 100 person positions.

In [ ]:
make_mag_animation(rirs_mp, sr, 'Moving Person', fps=10)

---
## 6. Animation — Room Layout with Moving Position

Animate the schematic room layout showing which position is active at each frame.

In [ ]:
def make_layout_animation(xs, ys, speaker_pos, mic_fixed_pos, scenario_label, fps=4):
    """Animate the room layout, highlighting the active position at each frame."""
    n = len(xs)
    fig_l, ax_l = plt.subplots(figsize=(5.5, 5))
    ax_l.set_xlim(0, 6)
    ax_l.set_ylim(0, 6)
    ax_l.set_aspect('equal')
    ax_l.add_patch(plt.Rectangle((0, 0), 6, 6, fill=False, edgecolor='black', linewidth=2))
    ax_l.plot(*speaker_pos, 'k^', markersize=12, label='Speaker')
    if mic_fixed_pos is not None:
        ax_l.plot(*mic_fixed_pos, 'bs', markersize=10, label='Microphone (fixed)')
    ax_l.scatter(xs, ys, c='lightgray', s=40, zorder=2)
    active_dot, = ax_l.plot([], [], 'ro', markersize=12, zorder=4, label='Active position')
    title_l = ax_l.set_title('')
    ax_l.set_xlabel('x [m] (schematic)')
    ax_l.set_ylabel('y [m] (schematic)')
    ax_l.legend(fontsize=8)
    ax_l.grid(True, linestyle=':', alpha=0.5)

    def init_l():
        active_dot.set_data([], [])
        return (active_dot,)

    def update_l(frame):
        active_dot.set_data([xs[frame]], [ys[frame]])
        title_l.set_text(f'{scenario_label}  —  Position {frame}')
        return (active_dot, title_l)

    ani_l = animation.FuncAnimation(
        fig_l, update_l, frames=n, init_func=init_l,
        interval=int(1000 / fps), blit=True,
    )
    plt.close(fig_l)
    return HTML(ani_l.to_jshtml())

print('Moving Listener — room layout animation')
make_layout_animation(ml_x, ml_y, speaker_pos, None, 'Moving Listener', fps=3)

In [ ]:
print('Moving Person — room layout animation')
make_layout_animation(mp_x, mp_y, speaker_pos, mic_fixed, 'Moving Person', fps=10)

---
## 7. Response Interpolation — Before and After

The adaptive EQ framework interpolates between adjacent RIRs during acoustic transitions. 
The cells below show the magnitude response evolution **without interpolation** (abrupt jumps between consecutive RIRs) and **with interpolation** (smooth crossfade), as implemented in `utils.common.interpolate_IRs`.

We use the first two moving-listener RIRs as a concrete example.

In [ ]:
# Build a sequence of frames: half at position 0, transition, half at position 1.
rir_a = torch.from_numpy(rirs_ml[0].astype(np.float32))
rir_b = torch.from_numpy(rirs_ml[1].astype(np.float32))

n_total_frames = 40       # total animation frames
transition_start = 10     # frame at which transition begins
transition_end   = 30     # frame at which transition ends

# Pad RIRs to the same length
max_len = max(len(rir_a), len(rir_b))
import torch.nn.functional as F
rir_a = F.pad(rir_a, (0, max_len - len(rir_a)))
rir_b = F.pad(rir_b, (0, max_len - len(rir_b)))

def frame_to_rir_no_interp(frame_idx):
    """Abrupt switch at transition_end."""
    if frame_idx < transition_end:
        return rir_a.numpy()
    return rir_b.numpy()

def frame_to_rir_with_interp(frame_idx):
    """Smooth crossfade during [transition_start, transition_end]."""
    if frame_idx <= transition_start:
        return rir_a.numpy()
    if frame_idx >= transition_end:
        return rir_b.numpy()
    t = (frame_idx - transition_start) / (transition_end - transition_start)
    # Use the interpolate_IRs function from common.py
    rirs_tensors = [rir_a.unsqueeze(0), rir_b.unsqueeze(0)]
    interp = interpolate_IRs(t, rirs_tensors[0], rirs_tensors[1])
    return interp.squeeze(0).numpy()

print(f'Transition: frames {transition_start} → {transition_end}  |  total frames: {n_total_frames}')

In [ ]:
def make_interp_animation(frame_to_rir_fn, title, nfft=NFFT, fps=8):
    """Animate magnitude response evolution with or without interpolation."""
    freqs_local = np.fft.rfftfreq(nfft, d=1.0 / sr)
    m = freqs_local > 50

    all_frames = [frame_to_rir_fn(i) for i in range(n_total_frames)]

    def get_smooth(rir):
        mag = compute_mag_db(rir, nfft=nfft)
        return log_smooth_curve(freqs_local[m], mag[m], window_pts=201)

    all_f = [get_smooth(r)[0] for r in all_frames]
    all_mag = [get_smooth(r)[1] for r in all_frames]
    y_min = min(np.min(m_) for m_ in all_mag) - 3
    y_max = max(np.max(m_) for m_ in all_mag) + 3

    fig_i, ax_i = plt.subplots(figsize=(8, 4))
    line_i, = ax_i.semilogx([], [], linewidth=1.5, color='darkorange')
    # Reference lines for RIR A and B
    f_a, mag_a = get_smooth(rir_a.numpy())
    f_b, mag_b = get_smooth(rir_b.numpy())
    ax_i.semilogx(f_a, mag_a, 'b--', linewidth=0.8, alpha=0.5, label='RIR 0 (start)')
    ax_i.semilogx(f_b, mag_b, 'r--', linewidth=0.8, alpha=0.5, label='RIR 1 (end)')
    ax_i.set_xlim(50, 20000)
    ax_i.set_ylim(y_min, y_max)
    ax_i.set_xlabel('Frequency [Hz]')
    ax_i.set_ylabel('Magnitude [dB]')
    ax_i.legend(fontsize=8)
    ax_i.grid(True, linestyle=':', alpha=0.6)
    title_i = ax_i.set_title('')

    def init_i():
        line_i.set_data([], [])
        return (line_i,)

    def update_i(frame):
        line_i.set_data(all_f[frame], all_mag[frame])
        region = 'before' if frame < transition_start else ('transitioning' if frame < transition_end else 'after')
        title_i.set_text(f'{title}  —  Frame {frame}  [{region}]')
        return (line_i, title_i)

    ani_i = animation.FuncAnimation(
        fig_i, update_i, frames=n_total_frames, init_func=init_i,
        interval=int(1000 / fps), blit=True,
    )
    plt.close(fig_i)
    return HTML(ani_i.to_jshtml())

print('Without interpolation — abrupt switch between positions')
make_interp_animation(frame_to_rir_no_interp, 'No interpolation')

In [ ]:
print('With interpolation — smooth crossfade between positions')
make_interp_animation(frame_to_rir_with_interp, 'With interpolation')

---
## 8. MedleyDB Input Signals

In [ ]:
from utils.common import discover_input_signals

medleydb_dir = root / 'data' / 'MedleyDB'
audio_files = sorted(medleydb_dir.glob('**/*.wav')) + sorted(medleydb_dir.glob('**/*.mp3'))
print(f'Found {len(audio_files)} audio files in {medleydb_dir}')
for f in audio_files:
    print(f'  {f.name}')

In [ ]:
if audio_files:
    input_cfg = {'use_songs_folder': True, 'max_num_songs': 4, 'max_audio_len_s': [30.0]}
    signals = discover_input_signals(input_cfg)
    print(f'Discovered {len(signals)} input signals.')

    fig, axes = plt.subplots(1, min(4, len(signals)), figsize=(14, 3), squeeze=False)
    for ax, (mode, info) in zip(axes[0], signals[:4]):
        wav, wav_sr = torchaudio.load(info['path'])
        if wav.shape[0] > 1:
            wav = wav.mean(0)
        else:
            wav = wav.squeeze(0)
        t = np.linspace(0, len(wav) / wav_sr, len(wav))
        ax.plot(t[:int(5*wav_sr)], wav[:int(5*wav_sr)].numpy(), linewidth=0.5)
        ax.set_title(Path(info['path']).stem[:30], fontsize=8)
        ax.set_xlabel('Time [s]')
        ax.set_ylabel('Amplitude')
    fig.suptitle('MedleyDB — first 5 seconds of up to 4 songs')
    fig.tight_layout()
    plt.show()
else:
    print('No MedleyDB audio files found. Place WAV/MP3 files in data/MedleyDB/.')

---
## 9. Target Response Preview

Preview the Kirkeby compensation target computed from the first RIR.

In [ ]:
from utils.common import get_compensation_EQ_params

rir = rirs_ml[0]
roi = (50.0, 20000.0)
eq_comp = get_compensation_EQ_params(rir, sr, ROI=roi, num_sections=7)

target_mag_resp  = eq_comp['target_response_db']
target_mag_freqs = eq_comp['freq_axis_smoothed']
rir_smooth_db    = eq_comp.get('rir_mag_db_smoothed', None)

fig, ax = plt.subplots(figsize=(9, 4))
if rir_smooth_db is not None:
    ax.semilogx(target_mag_freqs, rir_smooth_db, 'b--', linewidth=1.2, label='RIR (smoothed)')
ax.semilogx(target_mag_freqs, target_mag_resp, 'r-', linewidth=1.4, label='Compensation target')
ax.axhline(0, color='black', linewidth=0.8, linestyle=':')
ax.set_xlim(roi)
ax.set_xlabel('Frequency [Hz]')
ax.set_ylabel('Magnitude [dB]')
ax.set_title('Compensation target response (computed from first moving-listener RIR)')
ax.legend()
ax.grid(True, linestyle=':', alpha=0.7)
fig.tight_layout()
plt.show()